# 01 — Exploración de datos (EDA)

Este notebook explora la OBT `GOLD.OBT_CUSTOMER_CHURN` del PSet 2 para identificar decisiones que afectarán los experimentos de churn. La fuente es Snowflake; no se reconstruye la OBT desde CSV en este notebook.

## Alcance y protección contra fuga

Primero se inspecciona la tabla completa para revisar estructura y faltantes. Los análisis que requieren `Exited` —balance de clases, relación target-feature y atípicos para el modelado— usan únicamente `source_split == "train"`. Las filas `test` de la competencia no tienen target y no se usan como evaluación. La separación train/hold-out etiquetada se definirá después, una sola vez, en el paso de preparación.

Requisito: ejecutar antes el pipeline del PSet 2 y configurar las variables de Snowflake descritas en `README.md`.

In [ ]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "datos.py").exists():
    raise FileNotFoundError("Abre este notebook desde la carpeta PSet3.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.datos import load_obt

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
sns.set_theme(style="whitegrid", context="notebook")
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
obt = load_obt()
obt.columns = obt.columns.str.lower()

required_columns = {
    "observation_id", "source_split", "exited", "customer_id",
    "credit_score", "geography", "gender", "age", "tenure", "balance",
    "number_of_products", "has_credit_card", "is_active_member", "estimated_salary",
}
missing_columns = required_columns.difference(obt.columns)
if missing_columns:
    raise ValueError(f"La OBT no tiene las columnas esperadas: {sorted(missing_columns)}")

obt["source_split"] = obt["source_split"].astype("string").str.lower()
train = obt.loc[obt["source_split"].eq("train")].copy()
competition_test = obt.loc[obt["source_split"].eq("test")].copy()

if train.empty:
    raise ValueError("La OBT no contiene filas con source_split == 'train'.")
if train["exited"].isna().any():
    raise ValueError("Hay filas de train sin target Exited; revisar la carga o los modelos dbt.")
if not set(pd.to_numeric(train["exited"], errors="coerce").unique()).issubset({0, 1}):
    raise ValueError("Exited debe contener solamente las clases binarias 0 y 1 en train.")
if competition_test["exited"].notna().any():
    raise ValueError("El split test contiene Exited; revisar posible fuga desde la fuente.")

print(f"OBT total: {len(obt):,} filas × {obt.shape[1]} columnas")
print(f"Train etiquetado: {len(train):,} filas")
print(f"Test sin etiqueta: {len(competition_test):,} filas")

## 1. Tamaño, tipos y completitud

Se calcula la completitud para todas las columnas de la OBT. La etiqueta de test se espera ausente por definición; se reporta aparte para que esa ausencia no se confunda con un error de calidad.

In [ ]:
print("Dimensiones:", obt.shape)
display(pd.DataFrame({"tipo": obt.dtypes.astype(str), "no_nulos": obt.notna().sum()}))

missing = pd.DataFrame({
    "faltantes": obt.isna().sum(),
    "porcentaje_faltante": (100 * obt.isna().mean()).round(2),
}).sort_values("porcentaje_faltante", ascending=False)
display(missing)

target_missing_by_split = (
    obt.groupby("source_split", dropna=False)["exited"]
    .agg(total="size", con_target="count")
)
target_missing_by_split["target_faltante"] = (
    target_missing_by_split["total"] - target_missing_by_split["con_target"]
)
target_missing_by_split["pct_target_faltante"] = (
    100 * target_missing_by_split["target_faltante"] / target_missing_by_split["total"]
).round(2)
display(target_missing_by_split)

## 2. Estadística descriptiva

Los identificadores (`observation_id` y `customer_id`) se excluyen de la estadística de predictores: son claves, no magnitudes continuas. Se muestran estadísticas y conteos de categorías sobre train para mantener la interpretación ligada a las observaciones etiquetadas.

In [ ]:
numeric_features = [
    "credit_score", "age", "tenure", "balance", "number_of_products",
    "has_credit_card", "is_active_member", "estimated_salary",
]
categorical_features = ["geography", "gender", "age_band"]

for column in numeric_features + ["exited"]:
    train[column] = pd.to_numeric(train[column], errors="coerce")

display(train[numeric_features].describe().T)
for column in categorical_features:
    print(f"\n{column}: {train[column].nunique(dropna=False)} categorías")
    display(train[column].value_counts(dropna=False).rename("filas").to_frame())

## 3. Target y desbalance de clases

`Exited = 1` representa churn. El porcentaje de cada clase se calcula solo sobre train. La exactitud sola puede ser engañosa si una clase predomina; la elección de métrica técnica y de negocio se justificará en el diseño experimental con el costo del error definido para el proyecto.

In [ ]:
class_counts = train["exited"].value_counts().reindex([0, 1], fill_value=0)
target_distribution = pd.DataFrame({
    "filas": class_counts,
    "porcentaje": (100 * class_counts / class_counts.sum()).round(2),
})
target_distribution.index.name = "exited"
display(target_distribution)

churn_rate = float(train["exited"].mean())
majority_share = float(class_counts.max() / class_counts.sum())
minority_to_majority = float(class_counts.min() / class_counts.max())
print(f"Tasa de churn en train: {churn_rate:.2%}")
print(f"Baseline de exactitud por clase mayoritaria: {majority_share:.2%}")
print(f"Relación clase minoritaria/mayoritaria: {minority_to_majority:.3f}")

target_plot = target_distribution.reset_index()
target_plot["exited"] = target_plot["exited"].astype(str)
ax = sns.barplot(data=target_plot, x="exited", y="filas")
ax.set(title="Distribución de Exited en train", xlabel="Exited (0 = no churn, 1 = churn)", ylabel="Filas")
plt.show()

## 4. Distribuciones de predictores

Estas visualizaciones ayudan a identificar asimetrías y rangos para revisar. Un valor extremo no se elimina automáticamente: se contrasta con reglas de negocio y con la disponibilidad de la variable en inferencia.

In [ ]:
continuous_features = ["credit_score", "age", "tenure", "balance", "estimated_salary"]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
for column, ax in zip(continuous_features, axes.flat):
    sns.histplot(train[column].dropna(), bins=30, kde=True, ax=ax)
    ax.set_title(f"Distribución de {column}")
axes.flat[-1].axis("off")
plt.tight_layout()
plt.show()

## 5. Valores atípicos: IQR y Z-score

Se reportan dos señales complementarias en predictores numéricos continuos: fuera de `Q1 − 1.5 × IQR` / `Q3 + 1.5 × IQR`, y `|Z-score| ≥ 3`. Los indicadores no son juicios de error ni implican recorte.

In [ ]:
outlier_rows = []
outlier_masks = pd.DataFrame(index=train.index)
for column in continuous_features:
    values = train[column].dropna()
    q1, q3 = values.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    z = (train[column] - values.mean()) / values.std(ddof=0)
    iqr_mask = train[column].lt(lower) | train[column].gt(upper)
    z_mask = z.abs().ge(3)
    outlier_masks[column] = iqr_mask.fillna(False)
    outlier_rows.append({
        "feature": column,
        "iqr_lower": lower,
        "iqr_upper": upper,
        "iqr_outliers": int(iqr_mask.sum()),
        "iqr_outlier_pct": 100 * iqr_mask.mean(),
        "zscore_abs_ge_3": int(z_mask.sum()),
        "zscore_outlier_pct": 100 * z_mask.mean(),
    })

outlier_summary = pd.DataFrame(outlier_rows).set_index("feature").round(3)
display(outlier_summary)
print(f"Filas con al menos un predictor marcado por IQR: {outlier_masks.any(axis=1).sum():,} ({outlier_masks.any(axis=1).mean():.2%})")

fig, axes = plt.subplots(1, len(continuous_features), figsize=(18, 5))
for column, ax in zip(continuous_features, axes):
    sns.boxplot(y=train[column], ax=ax, color="steelblue")
    ax.set_title(column)
plt.tight_layout()
plt.show()

## 6. Relación entre target y características

Se usa Spearman para medir asociaciones monotónicas sin exigir normalidad y porque hay predictores ordinales/binarios y un target binario. La asociación no implica causalidad. Las correlaciones y las tasas categóricas se calculan solo con train.

In [ ]:
correlation_columns = numeric_features + ["exited"]
spearman = train[correlation_columns].corr(method="spearman")
target_association = (
    spearman["exited"].drop("exited")
    .to_frame("spearman_vs_exited")
    .assign(abs_spearman=lambda frame: frame["spearman_vs_exited"].abs())
    .sort_values("abs_spearman", ascending=False)
)
display(target_association.round(3))

plt.figure(figsize=(10, 7))
sns.heatmap(spearman, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1)
plt.title("Correlación de Spearman en train")
plt.tight_layout()
plt.show()

for column in categorical_features:
    rates = (
        train.groupby(column, dropna=False)["exited"]
        .agg(filas="size", tasa_churn="mean")
        .sort_values("tasa_churn", ascending=False)
    )
    rates["tasa_churn_pct"] = (100 * rates["tasa_churn"]).round(2)
    print(f"\nTasa de churn por {column}")
    display(rates[["filas", "tasa_churn_pct"]])

## 7. Hallazgos que afectan el modelado

La siguiente tabla genera evidencia directamente a partir del snapshot leído. Incluye al menos tres aspectos que afectan decisiones experimentales; revisar sus valores antes de redactar el memo. `CustomerId` es identificador y no se presenta como predictor.

In [ ]:
customer_groups = train.dropna(subset=["customer_id"]).groupby("customer_id", dropna=True)
duplicate_groups = customer_groups.size().gt(1)
n_duplicate_groups = int(duplicate_groups.sum())
duplicate_extra_rows = int((customer_groups.size() - 1).clip(lower=0).sum())
conflicting_target_groups = int(
    customer_groups["exited"].nunique().gt(1).sum()
)
invalid_customer_id_rows = int(train["customer_id"].isna().sum())

if not target_association.empty:
    strongest_feature = target_association.index[0]
    strongest_rho = float(target_association.iloc[0]["spearman_vs_exited"])
else:
    strongest_feature, strongest_rho = "N/D", float("nan")

if not outlier_summary.empty:
    highest_iqr_feature = outlier_summary["iqr_outlier_pct"].idxmax()
    highest_iqr_pct = float(outlier_summary.loc[highest_iqr_feature, "iqr_outlier_pct"])
else:
    highest_iqr_feature, highest_iqr_pct = "N/D", float("nan")

findings = pd.DataFrame([
    {
        "hallazgo": "Desbalance del target",
        "evidencia": f"Exited=1: {class_counts.loc[1]:,}/{class_counts.sum():,} ({churn_rate:.2%}); baseline mayoritaria={majority_share:.2%}.",
        "consecuencia": "Usar validación estratificada y no seleccionar modelos solo por accuracy; definir la métrica de negocio con el costo acordado en PSet 1.",
    },
    {
        "hallazgo": "CustomerId se repite en train",
        "evidencia": f"{n_duplicate_groups:,} grupos duplicados; {duplicate_extra_rows:,} filas adicionales; {conflicting_target_groups:,} grupos tienen ambas clases; {invalid_customer_id_rows:,} IDs nulos/no válidos.",
        "consecuencia": "No usar CustomerId como feature. Evaluar división por grupos (p. ej., StratifiedGroupKFold) para evitar que un cliente aparezca simultáneamente en entrenamiento y validación; tratar los IDs inválidos según la regla acordada.",
    },
    {
        "hallazgo": "Señales de valores atípicos",
        "evidencia": f"{highest_iqr_feature} tiene la mayor tasa IQR: {highest_iqr_pct:.2f}%; también se reporta |Z| ≥ 3 por feature.",
        "consecuencia": "Inspeccionar distribución y validez en inferencia; no eliminar ni recortar puntos sin evidencia de error y comparar modelos sensibles a outliers.",
    },
    {
        "hallazgo": "Asociación monotónica más alta con Exited",
        "evidencia": f"{strongest_feature}: rho de Spearman={strongest_rho:.3f}.",
        "consecuencia": "Considerarla para análisis de features, sin interpretarla como causalidad ni seleccionar el modelo usando el hold-out.",
    },
])
display(findings)

## Conclusiones y límites

- Revisar los conteos de faltantes: la ausencia de `Exited` en `test` es esperada, mientras faltantes en predictores deben investigarse.
- Confirmar con el PSet 1 el costo relativo de falsos negativos y falsos positivos antes de fijar la métrica de negocio y el umbral.
- Las correlaciones y diferencias de churn por grupo son asociaciones descriptivas, no efectos causales.
- Revisar la distribución de IDs repetidos y discutir el esquema de CV por grupo antes de comparar modelos.
- Esta EDA no ajusta imputación, escalado, codificación, selección de umbral ni modelos; esas operaciones se harán dentro del pipeline de entrenamiento.